# Estimacion de Pose Humana: Detectando Keypoints

La estimacion de pose detecta puntos clave (keypoints) del cuerpo humano: cabeza, hombros, codos, muñecas, caderas, rodillas, tobillos, etc.

**Aplicaciones:**
- Analisis deportivo (tecnica de movimientos)
- Rehabilitacion y fisioterapia
- Control de accesos (deteccion de personas)
- Videojuegos y realidad aumentada
- Sistemas de vigilancia

En este notebook usaremos modelos de Hugging Face para detectar poses de manera sencilla.

## Configuracion e Imports

In [ ]:
import torch
import transformers
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import requests
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version: {torch.__version__}, usando: {device}")
print(f"transformers version: {transformers.__version__}")

## Instalacion de Ultralytics (YOLOv8-Pose)

Vamos a usar YOLOv8-Pose que es uno de los modelos mas rapidos y precisos para pose estimation. No esta en transformers directamente, asi que usamos la libreria ultralytics.

**Nota**: Tambien existen modelos de pose en transformers como `microsoft/table-transformer-detection`, pero YOLOv8-Pose es mas especializado para este caso.

In [ ]:
# Instalar ultralytics si no esta instalado
try:
    from ultralytics import YOLO
except ImportError:
    !pip install ultralytics -q
    from ultralytics import YOLO

## Carga del Modelo YOLOv8-Pose

YOLOv8 tiene diferentes tamaños: n (nano), s (small), m (medium), l (large), x (extra large). Usaremos `yolov8m-pose.pt` que da buen balance entre velocidad y precision.

El modelo detecta **17 keypoints** del formato COCO:
0. Nariz
1. Ojo izquierdo
2. Ojo derecho
3. Oreja izquierda
4. Oreja derecha
5. Hombro izquierdo
6. Hombro derecho
7. Codo izquierdo
8. Codo derecho
9. Muñeca izquierda
10. Muñeca derecha
11. Cadera izquierda
12. Cadera derecha
13. Rodilla izquierda
14. Rodilla derecha
15. Tobillo izquierdo
16. Tobillo derecho

In [ ]:
# Cargar modelo (se descarga automaticamente la primera vez)
model = YOLO('yolov8m-pose.pt')
print("Modelo YOLOv8-Pose cargado")

## Carga de Imagenes de Ejemplo

In [ ]:
# Cargar imagen desde GitHub
url_person = "https://github.com/sergiovillanueva/prompt_to_mask/raw/master/assets/person_cars.jpg"
image_person = Image.open(BytesIO(requests.get(url_person).content)).convert("RGB")

print("Imagen cargada")
plt.imshow(image_person)
plt.axis("off")
plt.show()

## Detectar Pose en una Imagen

YOLOv8-Pose detecta automaticamente todas las personas en la imagen y sus keypoints.

In [ ]:
def detect_pose(image):
    """Detecta pose humana usando YOLOv8"""
    
    # Convertir PIL a numpy
    img_array = np.array(image)
    
    # Detectar poses
    results = model(img_array, verbose=False)
    
    # Dibujar keypoints y skeleton
    annotated_img = results[0].plot()
    
    # Mostrar resultados
    num_persons = len(results[0].keypoints)
    print(f"Detectadas {num_persons} personas\n")
    
    # Informacion de cada persona
    for i, keypoints in enumerate(results[0].keypoints):
        print(f"Persona {i+1}:")
        print(f"  Keypoints detectados: {keypoints.shape}")
        print(f"  Confianza promedio: {keypoints.conf.mean():.2f}")
    
    # Visualizar
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    axes[0].imshow(img_array)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Pose Detection ({num_persons} personas)")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return results

# Detectar pose
results = detect_pose(image_person)

## Visualizacion Personalizada de Keypoints

Podemos dibujar los keypoints de manera personalizada para resaltar partes especificas del cuerpo.

In [ ]:
def draw_keypoints_custom(image, results):
    """Dibuja keypoints con estilo personalizado"""
    
    img_array = np.array(image).copy()
    
    # Definir conexiones del skeleton (formato COCO)
    skeleton = [
        [15, 13], [13, 11], [16, 14], [14, 12], [11, 12],  # Piernas
        [5, 11], [6, 12],  # Torso
        [5, 6], [5, 7], [6, 8], [7, 9], [8, 10],  # Brazos
        [0, 1], [0, 2], [1, 3], [2, 4], [3, 5], [4, 6]  # Cabeza
    ]
    
    # Colores para diferentes partes
    colors = {
        'head': (255, 255, 0),    # Amarillo
        'arms': (0, 255, 255),    # Cyan
        'torso': (255, 0, 255),   # Magenta
        'legs': (0, 255, 0)       # Verde
    }
    
    for person_keypoints in results[0].keypoints:
        kpts = person_keypoints.xy[0].cpu().numpy()
        conf = person_keypoints.conf[0].cpu().numpy()
        
        # Dibujar lineas del skeleton
        for connection in skeleton:
            pt1_idx, pt2_idx = connection
            
            if conf[pt1_idx] > 0.5 and conf[pt2_idx] > 0.5:
                pt1 = tuple(map(int, kpts[pt1_idx]))
                pt2 = tuple(map(int, kpts[pt2_idx]))
                
                # Elegir color segun la parte del cuerpo
                if pt1_idx <= 4:  # Cabeza
                    color = colors['head']
                elif pt1_idx <= 10:  # Brazos
                    color = colors['arms']
                elif pt1_idx <= 12:  # Torso
                    color = colors['torso']
                else:  # Piernas
                    color = colors['legs']
                
                cv2.line(img_array, pt1, pt2, color, 3)
        
        # Dibujar keypoints
        for i, (kpt, c) in enumerate(zip(kpts, conf)):
            if c > 0.5:
                x, y = map(int, kpt)
                cv2.circle(img_array, (x, y), 5, (255, 0, 0), -1)  # Azul
    
    # Visualizar
    plt.figure(figsize=(10, 8))
    plt.imshow(img_array)
    plt.title("Pose con colores por parte del cuerpo")
    plt.axis("off")
    plt.show()

# Dibujar pose personalizada
draw_keypoints_custom(image_person, results)

## Analisis de Pose: Calcular Angulos

Podemos usar los keypoints para calcular angulos de articulaciones, util para analisis deportivo o fisioterapia.

In [ ]:
def calculate_angle(p1, p2, p3):
    """Calcula angulo entre tres puntos (p2 es el vertice)"""
    a = np.array(p1)
    b = np.array(p2)
    c = np.array(p3)
    
    ba = a - b
    bc = c - b
    
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))
    
    return angle

def analyze_pose(results):
    """Analiza angulos de las articulaciones"""
    
    for i, person_keypoints in enumerate(results[0].keypoints):
        kpts = person_keypoints.xy[0].cpu().numpy()
        conf = person_keypoints.conf[0].cpu().numpy()
        
        print(f"\nPersona {i+1} - Analisis de angulos:\n")
        
        # Angulo del codo derecho (hombro-codo-muñeca)
        if conf[6] > 0.5 and conf[8] > 0.5 and conf[10] > 0.5:
            angle_right_elbow = calculate_angle(kpts[6], kpts[8], kpts[10])
            print(f"  Codo derecho: {angle_right_elbow:.1f}°")
        
        # Angulo del codo izquierdo
        if conf[5] > 0.5 and conf[7] > 0.5 and conf[9] > 0.5:
            angle_left_elbow = calculate_angle(kpts[5], kpts[7], kpts[9])
            print(f"  Codo izquierdo: {angle_left_elbow:.1f}°")
        
        # Angulo de la rodilla derecha (cadera-rodilla-tobillo)
        if conf[12] > 0.5 and conf[14] > 0.5 and conf[16] > 0.5:
            angle_right_knee = calculate_angle(kpts[12], kpts[14], kpts[16])
            print(f"  Rodilla derecha: {angle_right_knee:.1f}°")
        
        # Angulo de la rodilla izquierda
        if conf[11] > 0.5 and conf[13] > 0.5 and conf[15] > 0.5:
            angle_left_knee = calculate_angle(kpts[11], kpts[13], kpts[15])
            print(f"  Rodilla izquierda: {angle_left_knee:.1f}°")

# Analizar pose
analyze_pose(results)

## Ejercicio: Detecta pose en otra imagen

Busca una imagen con personas en internet y prueba el detector de pose.

**Sugerencias:**
- Imagen de deportistas
- Personas haciendo yoga
- Foto grupal

In [ ]:
# EJERCICIO: Carga tu propia imagen y detecta pose

# Opcion 1: Desde URL
# tu_url = "https://..."  # <- Pon aqui la URL de tu imagen
# tu_imagen = Image.open(BytesIO(requests.get(tu_url).content)).convert("RGB")

# Opcion 2: Desde archivo local
# tu_imagen = Image.open("ruta/a/tu/imagen.jpg").convert("RGB")

# Detectar pose
# tu_results = detect_pose(tu_imagen)

# Analizar angulos
# analyze_pose(tu_results)


## Ejercicio Extra (Comodin): Contar personas

Crea una funcion que cuente automaticamente cuantas personas hay en una imagen grupal.

**Pista**: Ya tienes `len(results[0].keypoints)` que te da el numero de personas.

In [ ]:
# EJERCICIO EXTRA: Contador de personas
def count_people(image_url):
    """Cuenta personas en una imagen"""
    # Tu codigo aqui...
    pass

# Prueba con una imagen grupal:
# count_people("https://...")


## Resumen

En este notebook hemos aprendido:

✅ Que es la estimacion de pose y sus aplicaciones  
✅ Usar YOLOv8-Pose para detectar keypoints humanos  
✅ Los 17 keypoints del formato COCO  
✅ Visualizar poses con skeleton  
✅ Calcular angulos de articulaciones  
✅ Analisis de movimientos  

**Siguiente paso**: En el ultimo notebook veremos un showcase de otras tareas de vision: OCR, superresolucion, eliminacion de fondo, depth estimation y matching.

**Contacto**: Si tienes dudas puedes escribirme un email!